In [1]:
# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from pycircstat2.hypothesis import rayleigh_test
from mne.stats import permutation_t_test
from mne.stats import fdr_correction


ss = hf.settings_dict()

In [2]:
# Permutation settings
n_permutations = 2**ss['n_subjects']
alpha          = 0.05   # FDR threshold

results_dir = Path(ss["results_dir"])
results_dir.mkdir(parents=True, exist_ok=True)



In [4]:
# loop over each event type
for event_id in ss['event_id_list']:

    event_name = str(event_id)
    duty_cycle = ss['event_name_list'][event_id - 1]
    subjects_dir = ss['fs_subjects_dir']

    # load all participant z scores
    z_maps = []

    for subject_index in ss['subject_idx_list']:
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # load hilbert stc data
        hilbert_stc_file = Path(ss['hilbert_ref_dir']) / subject / event_name / f"{subject}-event-{event_name}-morph-z-vol.stc"
        stc = mne.read_source_estimate(hilbert_stc_file)

        z_maps.append(stc.data[:, 0])       # (n_voxels,)

    X = np.array(z_maps)                    # (12, n_voxels)

    print(f"  Data matrix       : {X.shape}  (subjects x voxels)")
    print(f"  Z min/max/mean    : {X.min():.3f} / {X.max():.3f} / {X.mean():.3f}")


    print(f"  Running voxel-wise permutation t-test ({n_permutations} permutations) ...")

    t_obs, p_val, H0 = permutation_t_test(
        X,
        n_permutations=n_permutations,
        tail=1,             # one-tailed: Z > 0
        n_jobs=-1,          # use all CPU cores
        verbose=True,
    )


    n_sig = (p_val < alpha).sum()
    print(f"  Significant voxels: {n_sig} / {len(p_val)}")
    print(f"  Brain activation percentage: {(n_sig/len(p_val))*100}%")


    def _make_stc(data_1d, template_stc):
        s        = template_stc.copy()
        s.data   = data_1d[:, np.newaxis]
        return s

    t_stc   = _make_stc(t_obs, stc)
    p_stc   = _make_stc(p_val, stc)

    event_dir = results_dir / event_name
    event_dir.mkdir(parents=True, exist_ok=True)

    t_file   = event_dir / f"group-event-{event_name}-tstat-fsaverage-vol"
    p_file   = event_dir / f"group-event-{event_name}-puncorr-fsaverage-vol"

    t_stc.save(str(t_file),     overwrite=True)
    p_stc.save(str(p_file),     overwrite=True)

    print(f"  T-map saved         : {t_file}-vl.stc")
    print(f"  p-map   : {p_file}-vl.stc")

print(f"\n{'='*60}")
print("Done.")




loading dataset for subject:  0005_3SJ
loading dataset for subject:  0002_TCZ
loading dataset for subject:  0009_YGZ
loading dataset for subject:  0010_ZMG
loading dataset for subject:  0011_MEE
loading dataset for subject:  0012_C3Z
loading dataset for subject:  0014_TAG
loading dataset for subject:  0015_QKW
loading dataset for subject:  0016_XLZ
loading dataset for subject:  0017_QJ5
loading dataset for subject:  0018_5T3
loading dataset for subject:  0019_COG
  Data matrix       : (12, 3031)  (subjects x voxels)
  Z min/max/mean    : -1.678 / 7.298 / 0.336
  Running voxel-wise permutation t-test (4096 permutations) ...
Permuting 4095 times (exact test)...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


  Significant voxels: 45 / 3031
  Brain activation percentage: 1.4846585285384362%
Writing STC to disk...
Overwriting existing file.
[done]
Writing STC to disk...
Overwriting existing file.
[done]
  T-map saved         : /home/elias/Workspace/thesis/iliasgdk_thesis/results/1/group-event-1-tstat-fsaverage-vol-vl.stc
  p-map   : /home/elias/Workspace/thesis/iliasgdk_thesis/results/1/group-event-1-puncorr-fsaverage-vol-vl.stc
loading dataset for subject:  0005_3SJ
loading dataset for subject:  0002_TCZ
loading dataset for subject:  0009_YGZ
loading dataset for subject:  0010_ZMG
loading dataset for subject:  0011_MEE
loading dataset for subject:  0012_C3Z
loading dataset for subject:  0014_TAG
loading dataset for subject:  0015_QKW
loading dataset for subject:  0016_XLZ
loading dataset for subject:  0017_QJ5
loading dataset for subject:  0018_5T3
loading dataset for subject:  0019_COG
  Data matrix       : (12, 3031)  (subjects x voxels)
  Z min/max/mean    : -3.024 / 7.887 / 0.293
  Runn

[Parallel(n_jobs=-1)]: Done   4 out of  12 | elapsed:    0.3s remaining:    0.6s
[Parallel(n_jobs=-1)]: Done   7 out of  12 | elapsed:    0.3s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done  10 out of  12 | elapsed:    0.4s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  12 out of  12 | elapsed:    0.4s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  12 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done   7 out of  12 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  10 out of  12 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  12 out of  12 | elapsed:    0.1s finished


  Significant voxels: 5 / 3031
  Brain activation percentage: 0.16496205872649292%
Writing STC to disk...
Overwriting existing file.
[done]
Writing STC to disk...
Overwriting existing file.
[done]
  T-map saved         : /home/elias/Workspace/thesis/iliasgdk_thesis/results/2/group-event-2-tstat-fsaverage-vol-vl.stc
  p-map   : /home/elias/Workspace/thesis/iliasgdk_thesis/results/2/group-event-2-puncorr-fsaverage-vol-vl.stc
loading dataset for subject:  0005_3SJ
loading dataset for subject:  0002_TCZ
loading dataset for subject:  0009_YGZ
loading dataset for subject:  0010_ZMG
loading dataset for subject:  0011_MEE
loading dataset for subject:  0012_C3Z
loading dataset for subject:  0014_TAG
loading dataset for subject:  0015_QKW
loading dataset for subject:  0016_XLZ
loading dataset for subject:  0017_QJ5
loading dataset for subject:  0018_5T3
loading dataset for subject:  0019_COG
  Data matrix       : (12, 3031)  (subjects x voxels)
  Z min/max/mean    : -1.985 / 6.759 / 0.289
  Runn

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  12 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done   7 out of  12 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  10 out of  12 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  12 out of  12 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  12 | elapsed:    0.1s remaining:    0.1s


  Significant voxels: 39 / 3031
  Brain activation percentage: 1.2867040580666447%
Writing STC to disk...
Overwriting existing file.
[done]
Writing STC to disk...
Overwriting existing file.
[done]
  T-map saved         : /home/elias/Workspace/thesis/iliasgdk_thesis/results/4/group-event-4-tstat-fsaverage-vol-vl.stc
  p-map   : /home/elias/Workspace/thesis/iliasgdk_thesis/results/4/group-event-4-puncorr-fsaverage-vol-vl.stc
loading dataset for subject:  0005_3SJ
loading dataset for subject:  0002_TCZ
loading dataset for subject:  0009_YGZ
loading dataset for subject:  0010_ZMG
loading dataset for subject:  0011_MEE
loading dataset for subject:  0012_C3Z
loading dataset for subject:  0014_TAG
loading dataset for subject:  0015_QKW
loading dataset for subject:  0016_XLZ
loading dataset for subject:  0017_QJ5
loading dataset for subject:  0018_5T3
loading dataset for subject:  0019_COG
  Data matrix       : (12, 3031)  (subjects x voxels)
  Z min/max/mean    : -2.128 / 7.482 / 0.320
  Runn

[Parallel(n_jobs=-1)]: Done   7 out of  12 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  10 out of  12 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  12 out of  12 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  12 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done   7 out of  12 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  10 out of  12 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  12 out of  12 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


  Significant voxels: 33 / 3031
  Brain activation percentage: 1.0887495875948532%
Writing STC to disk...
Overwriting existing file.
[done]
Writing STC to disk...
Overwriting existing file.
[done]
  T-map saved         : /home/elias/Workspace/thesis/iliasgdk_thesis/results/6/group-event-6-tstat-fsaverage-vol-vl.stc
  p-map   : /home/elias/Workspace/thesis/iliasgdk_thesis/results/6/group-event-6-puncorr-fsaverage-vol-vl.stc
loading dataset for subject:  0005_3SJ
loading dataset for subject:  0002_TCZ
loading dataset for subject:  0009_YGZ
loading dataset for subject:  0010_ZMG
loading dataset for subject:  0011_MEE
loading dataset for subject:  0012_C3Z
loading dataset for subject:  0014_TAG
loading dataset for subject:  0015_QKW
loading dataset for subject:  0016_XLZ
loading dataset for subject:  0017_QJ5
loading dataset for subject:  0018_5T3
loading dataset for subject:  0019_COG
  Data matrix       : (12, 3031)  (subjects x voxels)
  Z min/max/mean    : -2.087 / 4.745 / 0.071
  Runn

[Parallel(n_jobs=-1)]: Done   4 out of  12 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done   7 out of  12 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  10 out of  12 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  12 out of  12 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  12 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done   7 out of  12 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  10 out of  12 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  12 out of  12 | elapsed:    0.1s finished
